# Notebook 3 — Leakage-aware adversarial robustness evaluation

This notebook evaluates two existing models on a human-reviewed SMS adversarial set:

- **Normalized baseline**
- **Normalized + paraphrase augmentation**

The primary result excludes the seven exact training-overlap pairs listed in `excluded_training_overlap_pairs.csv`. An eligible pair must have a verified spam original, a human review of `yes` for meaning preservation, and a reviewed modified label of `spam`. An evasion occurs only when a model predicts the original as `spam` and its modified version as `ham`.

**Scope note:** This is a small, curated exploratory evaluation. The dataset has no timestamps, so it is not a temporal evaluation.

## 1. Install dependencies and check versions

In [1]:
!pip -q install pandas numpy scipy scikit-learn joblib matplotlib

import re
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn
from scipy.sparse import hstack

print("Python environment's scikit-learn version:", sklearn.__version__)
print("If joblib emits InconsistentVersionWarning, the models were saved with a different scikit-learn version.")
print("For a formal report, rerun with the model-saving version when available and confirm results are stable.")

Python environment's scikit-learn version: 1.6.1
If joblib emits InconsistentVersionWarning, the models were saved with a different scikit-learn version.
For a formal report, rerun with the model-saving version when available and confirm results are stable.


## 2. Check input files

In [3]:
ADV_PATH = Path('/content/validated_adversarial_predictions.csv')
OVERLAP_PATH = Path('/content/excluded_training_overlap_pairs.csv')
BASELINE_PATH = Path('/content/word_char_normalized_model.joblib')
AUGMENTED_PATH = Path('/content/word_char_normalized_paraphrase_augmented_model.joblib')

required_files = [ADV_PATH, OVERLAP_PATH, BASELINE_PATH, AUGMENTED_PATH]
missing_files = [str(p) for p in required_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Upload these files through the Colab Files sidebar:\n" + "\n".join(missing_files)
    )

for p in required_files:
    print(f"Found {p.name}: {p.stat().st_size / 1024**2:.2f} MB")

Found validated_adversarial_predictions.csv: 0.03 MB
Found excluded_training_overlap_pairs.csv: 0.00 MB
Found word_char_normalized_model.joblib: 10.38 MB
Found word_char_normalized_paraphrase_augmented_model.joblib: 10.38 MB


## 3. Load data and exclude training-overlap pairs

In [4]:
adv = pd.read_csv(ADV_PATH).fillna("")
overlap = pd.read_csv(OVERLAP_PATH).fillna("")

required_adv = {
    "original", "modified", "tactic", "verified_original_label",
    "meaning_preserved_review", "modified_true_label_review"
}
required_overlap = {"original", "modified"}
if missing := required_adv - set(adv.columns):
    raise ValueError(f"Reviewed adversarial CSV is missing columns: {sorted(missing)}")
if missing := required_overlap - set(overlap.columns):
    raise ValueError(f"Overlap CSV is missing columns: {sorted(missing)}")

def normalize_match_text(value):
    """Normalize whitespace only for exact pair matching; preserve message text otherwise."""
    return " ".join(str(value).strip().split())

overlap_keys = set(zip(
    overlap["original"].map(normalize_match_text),
    overlap["modified"].map(normalize_match_text)
))
adv_keys = list(zip(
    adv["original"].map(normalize_match_text),
    adv["modified"].map(normalize_match_text)
))
overlap_mask = pd.Series(adv_keys, index=adv.index).isin(overlap_keys)

adv_clean = adv.loc[~overlap_mask].copy()
print("Reviewed rows before exclusion:", len(adv))
print("Overlap pairs listed in file:", len(overlap))
print("Matching rows excluded:", int(overlap_mask.sum()))
print("Rows after exclusion:", len(adv_clean))

# Keep only explicitly reviewed, meaning-preserving spam pairs.
eligible = adv_clean[
    adv_clean["meaning_preserved_review"].astype(str).str.strip().str.lower().eq("yes")
    & adv_clean["modified_true_label_review"].astype(str).str.strip().str.lower().eq("spam")
    & adv_clean["verified_original_label"].astype(str).str.strip().str.lower().eq("spam")
].copy()

print("Leakage-aware eligible pairs:", len(eligible))
if eligible.empty:
    raise ValueError("No eligible pairs remain. Check the review labels and overlap file.")
display(eligible[["tactic", "original", "modified"]].head())

Reviewed rows before exclusion: 56
Overlap pairs listed in file: 7
Matching rows excluded: 7
Rows after exclusion: 49
Leakage-aware eligible pairs: 29


,tactic,original,modified
4,spacing,Online guidance by professional wa.me1+6_142.2...,o n l i n e guidance by professional wa.me1+6_...
6,charswap,Please confirm your details for shipment today...,"BPlease confim your etails for shipment tzday,..."
7,charswap,Once you've built up a bankroll hit me back an...,OnUce you've qbuilt up a bankrkoll hit me bakc...
8,charswap,"To celebrate 50 Million Users, we decided to h...","oT celebrate 50 MillioV UseIrs, we decided to ..."
10,charswap,rfqu2z Voicemail: You have 1 new Voicemail(s) ...,fqu2z Voicemail: Yoy have 1 neLw Voixemail(s) ...


## 4. Load models and verify bundle structure

In [5]:
model_paths = {
    "normalized_baseline": BASELINE_PATH,
    "normalized_augmented": AUGMENTED_PATH,
}
models = {}
expected_keys = {"word_vectorizer", "char_vectorizer", "classifier"}

for model_name, path in model_paths.items():
    bundle = joblib.load(path)
    if not isinstance(bundle, dict):
        raise TypeError(f"Expected a dictionary model bundle for {model_name}; got {type(bundle)}")
    missing = expected_keys - set(bundle.keys())
    if missing:
        raise ValueError(f"{model_name} is missing bundle keys: {sorted(missing)}")
    models[model_name] = bundle
    print(model_name, "loaded; keys:", sorted(bundle.keys()))

print("\nVersion compatibility reminder: if an InconsistentVersionWarning appeared above, these results should be confirmed using the scikit-learn version that saved the models.")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.9.1 when using version 1.6.1. This might lead to breaking code or in

normalized_baseline loaded; keys: ['char_vectorizer', 'classifier', 'normalization', 'word_vectorizer']
normalized_augmented loaded; keys: ['augmentation', 'char_vectorizer', 'classifier', 'normalization', 'word_vectorizer']

Version compatibility reminder: if an InconsistentVersionWarning appeared above, these results should be confirmed using the scikit-learn version that saved the models.


## 5. Predict and calculate leakage-aware evasion rates

In [6]:
def normalize_obfuscated_text(text):
    """Match the normalization rule used by the project training script."""
    s = str(text).lower()
    pattern = re.compile(r"(?<!\S)(?:[a-z0-9]\s+){3,}[a-z0-9](?!\S)", re.I)
    return pattern.sub(lambda m: re.sub(r"\s+", "", m.group(0)), s)

def predict_normalized_bundle(bundle, texts):
    texts = pd.Series(texts).fillna("").astype(str).tolist()
    normalized_texts = [normalize_obfuscated_text(t) for t in texts]
    word_features = bundle["word_vectorizer"].transform(normalized_texts)
    char_features = bundle["char_vectorizer"].transform(normalized_texts)
    features = hstack([word_features, char_features], format="csr")
    return bundle["classifier"].predict(features)

prediction_frames = []
for model_name, bundle in models.items():
    original_preds = predict_normalized_bundle(bundle, eligible["original"])
    modified_preds = predict_normalized_bundle(bundle, eligible["modified"])
    part = eligible.copy()
    part["model"] = model_name
    part["original_prediction"] = pd.Series(original_preds, index=part.index).astype(str).str.lower()
    part["modified_prediction"] = pd.Series(modified_preds, index=part.index).astype(str).str.lower()
    part["original_detected_spam"] = part["original_prediction"].eq("spam")
    part["evasion_success"] = part["original_detected_spam"] & part["modified_prediction"].eq("ham")
    prediction_frames.append(part)

predictions = pd.concat(prediction_frames, ignore_index=True)

def summarize(group, model_name=None, tactic=None):
    denom = int(group["original_detected_spam"].sum())
    successes = int(group["evasion_success"].sum())
    return {
        "model": model_name if model_name is not None else group["model"].iloc[0],
        **({"tactic": tactic} if tactic is not None else {}),
        "eligible_pairs": int(len(group)),
        "originals_predicted_spam": denom,
        "evasion_successes": successes,
        "evasion_rate_percent": round(100 * successes / denom, 2) if denom else np.nan,
    }

overall_results = pd.DataFrame([
    summarize(predictions[predictions["model"].eq(name)], model_name=name)
    for name in models
])
tactic_results = pd.DataFrame([
    summarize(group, model_name=model_name, tactic=tactic)
    for (model_name, tactic), group in predictions.groupby(["model", "tactic"], dropna=False)
])

print("PRIMARY RESULT — LEAKAGE-AWARE OVERALL COMPARISON")
display(overall_results)
print("LEAKAGE-AWARE RESULTS BY TACTIC")
display(tactic_results.sort_values(["tactic", "model"]).reset_index(drop=True))

PRIMARY RESULT — LEAKAGE-AWARE OVERALL COMPARISON


,model,eligible_pairs,originals_predicted_spam,evasion_successes,evasion_rate_percent
0,normalized_baseline,29,28,0,0.0
1,normalized_augmented,29,28,0,0.0


LEAKAGE-AWARE RESULTS BY TACTIC


,model,tactic,eligible_pairs,originals_predicted_spam,evasion_successes,evasion_rate_percent
0,normalized_augmented,charswap,5,4,0,0.0
1,normalized_baseline,charswap,5,4,0,0.0
2,normalized_augmented,hybrid,5,5,0,0.0
3,normalized_baseline,hybrid,5,5,0,0.0
4,normalized_augmented,paraphrasing,3,3,0,0.0
5,normalized_baseline,paraphrasing,3,3,0,0.0
6,normalized_augmented,punycode,6,6,0,0.0
7,normalized_baseline,punycode,6,6,0,0.0
8,normalized_augmented,spacing,10,10,0,0.0
9,normalized_baseline,spacing,10,10,0,0.0


## 6. Export final tables

In [7]:
OUT = Path("/content/sms_spam_outputs")
OUT.mkdir(parents=True, exist_ok=True)

eligible.to_csv(OUT / "leakage_aware_eligible_pairs.csv", index=False)
predictions.to_csv(OUT / "leakage_aware_predictions.csv", index=False)
overall_results.to_csv(OUT / "normalized_leakage_aware_overall.csv", index=False)
tactic_results.to_csv(OUT / "normalized_leakage_aware_tactics.csv", index=False)

print("Saved outputs to:", OUT)
for p in sorted(OUT.glob("*leakage_aware*.csv")):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")

Saved outputs to: /content/sms_spam_outputs
 - leakage_aware_eligible_pairs.csv (14,026 bytes)
 - leakage_aware_predictions.csv (30,081 bytes)
 - normalized_leakage_aware_overall.csv (150 bytes)
 - normalized_leakage_aware_tactics.csv (493 bytes)


## 7. Interpretation and reporting cautions

- Treat the leakage-aware comparison above as the **primary adversarial result**.
- An evasion is counted only when the original is predicted as spam and the reviewed modified spam is predicted as ham.
- Report denominators alongside rates; the evaluation set is small and curated.
- If both models have the same evasion rate, this experiment does not demonstrate an incremental benefit from paraphrase augmentation.
- The model bundles were previously reported as saved with scikit-learn 1.9.1, while some Colab sessions used 1.6.1. If a version warning appears, rerun in a compatible environment and verify the predictions before treating them as final.
- Have a human independently validate all labels and paraphrase pairs before making formal claims. Excluding exact pairs does not rule out all near-duplicate or source-level leakage.
- The canonical dataset has no timestamps. Do not claim temporal evaluation unless a timestamped dataset and a time-based split are used.